# 사전학습: 재료 데이터를 읽고 조회하기
AI for Materials Science — Hands-on session 1

수업에서 바로 쓰게 될 도구를 미리 손에 익히는 시간입니다.
숫자 묶음을 다루는 NumPy, 표를 다루는 pandas, 재료의 조성과 구조를 다루는 pymatgen을 차례로 보고,
마지막에는 재료 데이터베이스에 직접 자료를 요청해 보겠습니다.

<table style="width:100%; font-size:1.15em; border-collapse:collapse; margin:1em 0;">
<thead>
<tr>
<th style="width:6%; padding:12px 16px; text-align:center;">절</th>
<th style="padding:12px 16px; text-align:left;">할 일</th>
<th style="width:18%; padding:12px 16px; text-align:right;">학습 시간</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding:12px 16px; text-align:center;">0</td>
<td style="padding:12px 16px;">준비: 설치와 자료 내려받기</td>
<td style="padding:12px 16px; text-align:right;">3분</td>
</tr>
<tr>
<td style="padding:12px 16px; text-align:center;">1</td>
<td style="padding:12px 16px;">NumPy 배열과 조건 선택</td>
<td style="padding:12px 16px; text-align:right;">6분</td>
</tr>
<tr>
<td style="padding:12px 16px; text-align:center;">2</td>
<td style="padding:12px 16px;">CSV 읽기, 결측값 확인, 후보 선택</td>
<td style="padding:12px 16px; text-align:right;">9분</td>
</tr>
<tr>
<td style="padding:12px 16px; text-align:center;">3</td>
<td style="padding:12px 16px;">화학식과 결정 구조, CIF 읽기·쓰기</td>
<td style="padding:12px 16px; text-align:right;">8분</td>
</tr>
<tr>
<td style="padding:12px 16px; text-align:center;">4</td>
<td style="padding:12px 16px;">키 없이 OQMD 조회</td>
<td style="padding:12px 16px; text-align:right;">5분</td>
</tr>
<tr>
<td style="padding:12px 16px; text-align:center;">5</td>
<td style="padding:12px 16px;">(선택) MP에서 LiFePO₄ 조회</td>
<td style="padding:12px 16px; text-align:right;">+3분</td>
</tr>
</tbody>
</table>

위에서부터 한 셀씩 차례로 실행해주세요.
앞 셀에서 만든 변수를 뒤 셀에서 그대로 쓰기 때문에, 순서를 건너뛰면 "이름을 찾을 수 없다"는 오류가 납니다.
코드 안의 `##` 줄은 여러분에게 설명하려고 적어 둔 주석이라 실행되지 않습니다.

## 0. 준비

시작하기 전에 두 가지를 갖춰 두겠습니다. 하나는 코드를 돌릴 라이브러리, 다른 하나는 읽어 들일 데이터입니다.

### 0-1. 라이브러리 설치
`pymatgen` 하나만 설치하면 됩니다.
NumPy, pandas, requests는 pymatgen이 내부에서 쓰는 라이브러리라 함께 따라 들어옵니다.
설치에 1~2분 정도 걸리니 셀을 실행해 두고 아래 설명을 먼저 읽어도 좋습니다.

In [ ]:
## 맨 앞의 !는 파이썬이 아니라 터미널 명령을 실행하라는 표시입니다. -q는 설치 과정을 조용히 넘기는 옵션입니다.
!pip install -q pymatgen

### 0-2. 수업 자료 내려받기
실습에 쓸 CSV, CIF, 그리고 미리 받아 둔 OQMD 응답 사본은 수업 저장소의 `Data/` 폴더에 들어 있습니다.
아래 셀을 실행하면 저장소가 통째로 현재 작업 폴더에 복사됩니다.
개인 컴퓨터의 폴더 주소를 적거나 Google Drive를 연결할 필요는 없습니다.

이미 한 번 받은 뒤에 다시 실행하면 "폴더가 이미 있다"는 메시지가 나오는데, 문제가 아니니 그대로 다음 셀로 넘어가세요.

In [ ]:
## git clone은 GitHub에 있는 저장소를 통째로 내려받는 명령입니다.
!git clone https://github.com/kwongibaek/MS49900-AI4M.git

### 0-3. 라이브러리 불러오기
설치와 `import`는 다른 단계입니다.
설치는 컴퓨터 어딘가에 파일을 놓아두는 일이고, `import`는 그 도구를 지금 이 노트북 안으로 가져오는 일입니다.
그래서 설치를 했더라도 `import`를 하지 않으면 이름을 찾을 수 없다는 오류가 납니다.

`as`는 긴 이름 대신 쓸 짧은 별명을 붙여줍니다. `numpy`를 매번 적는 대신 `np`라고 쓰는 식입니다.

In [ ]:
## 파일 경로와 JSON을 다루는 파이썬 기본 도구입니다.
import json
import os
from pathlib import Path

## np는 숫자 배열, pd는 표, requests는 인터넷 요청, display는 표를 보기 좋게 출력할 때 씁니다.
import numpy as np
import pandas as pd
import requests
from IPython.display import display

### 0-4. 파일 경로 정하기
읽어야 할 파일이 어디 있는지 여기서 한 번만 정해 두고, 뒤에서는 `STEEL_CSV` 같은 짧은 이름으로 부르겠습니다.
이렇게 해두면 Colab에서 열든 내려받은 저장소에서 열든 아래 코드를 고치지 않고 그대로 쓸 수 있습니다.

셀을 실행하면 찾아낸 폴더 위치와 그 안의 파일 목록이 나옵니다.

In [ ]:
## 노트북을 어디서 열었는지에 따라 Data 폴더의 위치가 달라집니다.
## 세 후보를 차례로 확인해서, 실제로 존재하는 폴더를 DATA_DIR에 담습니다.
for candidate in [Path("MS49900-AI4M/Data"), Path("Data"), Path("../Data")]:
    if candidate.is_dir():
        DATA_DIR = candidate
        break

## /는 폴더 이름과 파일 이름을 이어 붙이는 기호입니다.
STEEL_CSV = DATA_DIR / "steel_strength.csv"
LFP_CIF = DATA_DIR / "LiFePO4.cif"
OQMD_CACHE = DATA_DIR / "oqmd_li_fe_p_o_contains_sample.json"

## 실습에서 만든 파일은 여기에 저장하겠습니다. 폴더가 없으면 새로 만듭니다.
OUTPUT_DIR = Path("outputs/01_preclass")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(DATA_DIR)
sorted(path.name for path in DATA_DIR.iterdir())

## 1. NumPy: 배열에서 필요한 값 고르기 · 6분

재료 데이터는 결국 숫자가 잔뜩 모인 형태입니다. 이런 숫자 묶음을 다루는 도구가 NumPy의 **배열(array)**입니다.

파이썬 리스트와 비슷해 보이지만, 값을 하나씩 반복문으로 돌지 않고 묶음 전체를 한 번에 계산할 수 있다는 점이 다릅니다.
데이터가 커질수록 이 차이가 크게 벌어집니다.

여기서는 항복강도 다섯 개를 예로 들어 배열을 만들고, 원하는 값만 골라내는 데까지 연습하겠습니다.
실제 실험 자료는 다음 절에서 읽습니다.

### 1-1. 배열 만들기
`np.array()`에 숫자 목록을 넣으면 배열이 만들어집니다.

출력의 `dtype`은 값이 어떤 형태로 저장됐는지 알려줍니다. 여기서는 소수점을 가진 실수라 `float64`가 나옵니다.
`shape`는 배열의 크기인데, `(5,)`는 값 다섯 개가 한 줄로 놓여 있다는 뜻입니다.

In [ ]:
## np는 0-3에서 붙여 둔 NumPy의 별명입니다. = 오른쪽에서 만든 배열을 왼쪽 이름에 담습니다.
strength = np.array([620.0, 780.0, 910.0, 1050.0, 1180.0])

print("배열:", strength)
print("자료형과 모양:", strength.dtype, strength.shape)

### 1-2. 위치로 값 꺼내기
배열에서 값 하나를 꺼내려면 대괄호 안에 위치를 적습니다.
주의할 점은 파이썬이 위치를 **0부터** 센다는 것입니다. 그래서 첫 값이 `[1]`이 아니라 `[0]`입니다.

`[-1]`처럼 음수를 쓰면 뒤에서부터 셉니다. 마지막 값을 꺼낼 때 자주 씁니다.
콜론을 쓴 `[1:4]`는 범위를 고르는데, 시작은 포함하고 끝은 포함하지 않아 위치 1, 2, 3의 값 세 개가 나옵니다.

In [ ]:
print("첫 값:", strength[0])
print("마지막 값:", strength[-1])
print("일부 값:", strength[1:4])

### 1-3. 배열 전체에 같은 계산 적용하기
배열에 숫자 하나를 더하면 모든 값에 그 숫자가 한 번에 더해집니다. 반복문을 쓸 필요가 없습니다.
측정값을 보정하거나 단위를 바꿀 때 이런 방식을 자주 씁니다.

참고로 셀의 마지막 줄에 변수 이름만 적어 두면 `print()` 없이도 그 값이 결과로 표시됩니다.

In [ ]:
## 원래 strength는 그대로 두고, 25를 더한 새 배열을 adjusted에 담습니다.
adjusted = strength + 25.0
adjusted

### 1-4. 조건으로 값 고르기
`>=` 같은 비교를 배열에 적용하면 결과가 True/False 하나가 아니라 **값마다 판정한 True/False 배열**로 나옵니다.
이것을 마스크(mask)라고 부릅니다.

원하는 값을 골라내는 작업은 대부분 이 마스크를 거칩니다.
먼저 마스크가 어떻게 생겼는지부터 확인해 보겠습니다.

In [ ]:
## 값 다섯 개를 각각 900과 비교한 결과가 나옵니다.
mask = strength >= 900.0
mask

이제 이 마스크로 실제 값을 골라내겠습니다.
대괄호에 마스크를 넣으면 True인 자리의 값만 남고, 남은 값들로 개수나 평균을 바로 구할 수 있습니다.

In [ ]:
selected = strength[mask]

## size는 값의 개수, mean()은 평균입니다.
print("선택된 값:", selected)
print("선택된 개수와 평균:", selected.size, selected.mean())

## 2. pandas: CSV를 읽고 강재 후보 고르기 · 9분

이번에는 실제 실험 데이터를 다뤄 보겠습니다.
`steel_strength.csv`에는 강재 312종의 조성과 측정 물성이 들어 있습니다.
원소 함량은 wt%, 항복강도와 인장강도는 MPa, 연신율은 %입니다.

이렇게 행과 열로 된 표를 pandas에서는 **DataFrame**이라고 부릅니다.
엑셀 시트를 파이썬 안에서 다룬다고 생각하면 이해하기 쉽습니다.

목표는 "항복강도가 높으면서도 잘 늘어나는 강재"를 조건으로 걸러내는 것입니다.
읽기 → 살펴보기 → 조건 걸기 → 저장하기 순서로 진행하겠습니다.

### 2-1. CSV 읽기
`pd.read_csv()`에 파일 경로를 주면 표를 읽어 옵니다.
`head()`는 앞 다섯 행만 보여주는데, 데이터를 처음 열었을 때 열 이름과 값의 생김새를 확인하는 습관을 들이면 좋습니다.

In [ ]:
steel = pd.read_csv(STEEL_CSV)
steel.head()

표가 얼마나 큰지도 확인해 봅시다.
`shape`는 (행 수, 열 수)를 한 번에 보여주고, `len()`은 행 수만 알려줍니다.

In [ ]:
print("행과 열:", steel.shape)
print("전체 행 수:", len(steel))

### 2-2. 필요한 열만 고르기
열 이름 하나를 대괄호에 넣으면 그 열만, 이름 목록을 넣으면 여러 열을 한꺼번에 고를 수 있습니다.
열이 17개나 되니 지금 관심 있는 물성 세 개만 따로 떼어 두겠습니다.

In [ ]:
## 열 이름 세 개를 리스트로 묶어 한 번에 고릅니다.
property_columns = ["yield strength", "tensile strength", "elongation"]
properties = steel[property_columns]
properties.head()

### 2-3. 결측값 확인하기
실험 데이터에는 측정하지 못한 칸이 섞여 있기 마련입니다. pandas는 이런 빈칸을 `NaN`으로 표시합니다.

조건을 걸기 전에 어느 열에 빈칸이 몇 개 있는지 먼저 세어 보겠습니다.
연신율 열에만 빈칸이 있다는 점을 기억해 두세요. 바로 다음 단계에서 이것 때문에 처리를 하나 더 하게 됩니다.

In [ ]:
## count()는 값이 들어 있는 칸을, isna().sum()은 비어 있는 칸을 열별로 셉니다.
pd.DataFrame({
    "값이 있는 개수": properties.count(),
    "결측값 개수": properties.isna().sum(),
})

### 2-4. 조건 두 개 만들기
"항복강도 1500 MPa 이상"과 "연신율 10% 이상"을 각각 조건으로 만들겠습니다.
1-4의 마스크와 원리가 똑같습니다. 조건 하나가 행마다 True/False를 담은 목록이 됩니다.

연신율 쪽에는 `notna()`를 함께 걸었습니다.
빈칸은 어차피 비교에서 False로 처리되지만, "값이 있고 그 값이 10 이상"이라는 뜻을 코드에 분명히 남겨 두면
나중에 다시 읽을 때 의도를 헷갈리지 않습니다.

In [ ]:
## 조건 하나하나가 312개 행 전체에 대한 True/False 목록입니다.
strength_condition = steel["yield strength"] >= 1500.0
elongation_condition = steel["elongation"].notna() & (steel["elongation"] >= 10.0)

## True를 1로 세면 각 조건을 통과한 행이 몇 개인지 알 수 있습니다.
print("강도 조건 통과:", int(strength_condition.sum()))
print("연신율 조건 통과:", int(elongation_condition.sum()))

### 2-5. 두 조건을 모두 만족하는 행 고르기
`&`로 두 조건을 이으면 둘 다 True인 행만 남습니다.
각각을 통과한 개수보다 크게 줄어드는 것을 확인해 보세요. 조건을 겹칠수록 후보가 빠르게 좁아집니다.

`loc[조건]`이 실제로 행을 골라내는 부분이고, `sort_values()`로 강도가 높은 순서대로 정렬했습니다.

In [ ]:
## copy()는 원본 steel과 분리된 새 표를 만듭니다. 뒤에서 값을 바꿔도 원본이 영향받지 않습니다.
candidates = steel.loc[strength_condition & elongation_condition].copy()
candidates = candidates.sort_values("yield strength", ascending=False)

print("두 조건 모두 통과:", len(candidates))
display(candidates[["formula", *property_columns]].head())

### 2-6. 결과 저장하기
골라낸 후보를 파일로 남겨 두면 다음 수업이나 보고서에서 다시 쓸 수 있습니다.
CSV는 `outputs/01_preclass/` 폴더에 만들어집니다.

In [ ]:
## index=False로 두면 pandas가 임시로 붙인 행 번호가 CSV에 별도 열로 들어가지 않습니다.
candidate_csv = OUTPUT_DIR / "steel_candidates.csv"
candidates.to_csv(candidate_csv, index=False)

print("저장한 파일:", candidate_csv)

## 3. pymatgen: 화학식과 결정 구조 다루기 · 8분

앞 절까지는 재료를 숫자 표로만 봤습니다. 이제 재료 자체를 다루는 도구를 써 보겠습니다.

pymatgen에는 성격이 다른 두 객체가 있습니다.

- `Composition`은 "무엇이 얼마나 들어 있는가", 즉 조성만 다룹니다.
- `Structure`는 여기에 격자와 원자 위치까지 더해 결정 구조 전체를 담습니다.

LiFePO₄ 화학식을 해석해 보고, 실리콘 결정 구조를 직접 만들어 CIF 파일로 저장한 뒤,
준비된 LiFePO₄ CIF를 다시 읽어 보겠습니다.

참고로 실험으로 얻은 결정 구조를 찾을 때 ICSD는 보통 이용 라이선스가 필요하고, COD는 누구나 접근할 수 있습니다.
여기서 읽는 CIF는 pymatgen이 예제로 공개한 파일입니다.

### 3-1. 화학식 해석하기
`Composition("LiFePO4")`처럼 화학식을 문자열로 넣으면 pymatgen이 원소와 개수를 알아서 해석합니다.
사람이 쓰는 표기를 프로그램이 계산할 수 있는 형태로 바꿔 주는 셈입니다.

In [ ]:
## 이번 절에서 쓸 세 가지를 pymatgen에서 가져옵니다.
from pymatgen.core import Composition, Lattice, Structure

lfp_composition = Composition("LiFePO4")

## reduced_formula는 가장 간단한 정수비로 줄인 화학식, num_atoms는 화학식 단위 하나에 든 원자 수입니다.
print("간단한 화학식:", lfp_composition.reduced_formula)
print("화학식의 원자 수 합:", lfp_composition.num_atoms)

원자 분율은 각 원소의 원자 수를 전체 원자 수로 나눈 값입니다.
LiFePO₄는 원자 7개 중 산소가 4개라 산소 분율이 4/7 ≈ 0.571로 나옵니다.

In [ ]:
lfp_composition.fractional_composition.as_dict()

### 3-2. 대칭으로 구조 만들기
결정 구조를 만들 때 원자 위치를 전부 적을 필요는 없습니다.
공간군이 정해지면 대표 원자 하나의 위치에서 나머지 자리가 대칭으로 자동 결정되기 때문입니다.

실리콘은 다이아몬드 구조(공간군 Fd-3m)이고 격자상수는 5.431 Å입니다.
원자 위치를 하나만 넣었는데도 단위 셀 안의 원자 자리가 8개로 채워지는지 확인해 보세요.

In [ ]:
## species와 coords에는 대표 원자 하나만 넣습니다. 나머지 자리는 Fd-3m 대칭이 만들어 줍니다.
silicon = Structure.from_spacegroup(
    "Fd-3m", Lattice.cubic(5.431), species=["Si"], coords=[[0, 0, 0]],
)

## abc는 격자의 세 변 길이이고 단위는 Å입니다.
print("격자 길이:", silicon.lattice.abc)
print("이 셀의 원자 자리 수:", len(silicon))

### 3-3. CIF 파일로 저장하기
CIF는 결정 구조를 주고받을 때 표준으로 쓰는 형식입니다.
파일로 저장해 두면 VESTA 같은 시각화 프로그램이나 계산 코드에서도 그대로 읽을 수 있습니다.

In [ ]:
## fmt="cif"로 저장 형식을 지정합니다.
silicon_cif = OUTPUT_DIR / "silicon.cif"
silicon.to(filename=str(silicon_cif), fmt="cif")

print("저장한 구조:", silicon_cif)

### 3-4. CIF 파일 읽기
저장이 되면 반대로 읽어 오는 것도 됩니다. 이번에는 준비된 LiFePO₄ CIF를 읽어 보겠습니다.

3-1에서는 화학식만 다뤘지만 여기서는 격자 크기와 원자 개수까지 함께 나옵니다.
원자 자리가 28개인 이유는 이 파일의 단위 셀이 화학식 단위 4개로 이루어져 있기 때문입니다(7 × 4 = 28).

In [ ]:
## from_file()은 파일을 읽어 Structure 객체로 만들어 줍니다.
lfp_structure = Structure.from_file(LFP_CIF)

print("화학식:", lfp_structure.composition.reduced_formula)
print("격자 길이:", lfp_structure.lattice.abc)
print("원자 자리 수:", len(lfp_structure))

구조 안의 원자 하나하나에도 접근할 수 있습니다.
분율 좌표는 격자 벡터를 기준으로 위치를 0에서 1 사이 값으로 나타낸 것입니다. Å 단위 거리가 아니라는 점에 주의하세요.

In [ ]:
## [0]은 첫 번째 원자 자리입니다.
print("첫 원소:", lfp_structure[0].species_string)
print("첫 원자의 분율 좌표:", lfp_structure[0].frac_coords)

## 4. OQMD: API 키 없이 계산 자료 조회하기 · 5분

지금까지는 미리 받아 둔 파일을 읽었습니다. 이번에는 데이터베이스 서버에 직접 자료를 요청해 보겠습니다.

API는 "이런 조건의 자료를 주세요"라고 주소와 조건을 보내면
프로그램이 읽을 수 있는 형태로 답을 돌려주는 창구라고 생각하면 됩니다.
OQMD는 별도 키 없이 열려 있어 첫 연습으로 좋습니다.

Li, Fe, P, O를 모두 포함하는 자료를 최대 20건 받아 보겠습니다.
**조건 만들기 → 요청 보내기 → 응답을 표로 바꾸기** 세 단계로 나눠서 진행합니다.

표에 나올 `delta_e`는 원자당 형성 에너지, `stability`는 convex hull까지의 거리이며 둘 다 eV/atom 단위의 계산값입니다.
다만 계산값은 기준 에너지와 보정 방법이 데이터베이스마다 달라서, OQMD 값과 MP 값을 행별로 그대로 비교하면 안 됩니다.

### 4-1. 검색 조건 만들기
요청에 담을 내용을 먼저 딕셔너리로 정리하겠습니다.
`fields`는 받아올 항목, `filter`는 어떤 자료를 고를지 정하는 조건, `limit`은 최대 몇 건까지 받을지입니다.

`element_set=(Li,Fe,P,O)`는 네 원소를 **포함**하라는 뜻이라 다른 원소가 더 들어 있어도 걸립니다.
정확히 네 원소만 원한다면 조건에 `AND ntypes=4`를 덧붙이면 됩니다.

In [ ]:
OQMD_URL = "https://oqmd.org/oqmdapi/formationenergy"

## 응답에서 받아올 항목 이름입니다.
OQMD_FIELDS = ["name", "entry_id", "spacegroup", "ntypes", "natoms",
               "delta_e", "stability", "band_gap", "prototype", "icsd_id"]

## join은 항목 목록을 쉼표로 이어 하나의 문자열로 만듭니다.
OQMD_PARAMS = {
    "fields": ",".join(OQMD_FIELDS),
    "limit": 20,
    "offset": 0,
    "format": "json",
    "filter": "element_set=(Li,Fe,P,O)",
}

OQMD_PARAMS

### 4-2. 요청 보내기
`requests.get()`이 실제로 서버에 요청을 보내는 부분입니다.
`raise_for_status()`는 서버가 오류로 답했을 때 그냥 넘어가지 않고 바로 알려주는 안전장치입니다.

수업 시간에 여러 명이 동시에 접속하면 OQMD 서버가 느려질 수 있습니다.
그래서 요청이 실패하면 저장소에 함께 담아 둔 같은 조건의 응답 사본으로 자동 전환되도록 해두었습니다.
출력의 "데이터 출처"에서 어느 쪽이 쓰였는지만 확인하고 그대로 진행하면 됩니다.

In [ ]:
try:
    ## 여기서 실제로 인터넷 요청이 나갑니다. timeout은 30초까지만 기다리라는 뜻입니다.
    response = requests.get(OQMD_URL, params=OQMD_PARAMS, timeout=30)
    response.raise_for_status()
    ## 서버는 JSON 문자열로 답합니다. .json()이 이를 파이썬 딕셔너리로 바꿔 줍니다.
    oqmd_payload = response.json()
    oqmd_source = "OQMD 실시간 HTTP 응답"
except requests.RequestException as error:
    ## 요청이 실패했을 때만 이 부분이 실행되어, 미리 받아 둔 파일을 대신 읽습니다.
    oqmd_payload = json.loads(OQMD_CACHE.read_text(encoding="utf-8"))
    oqmd_source = f"OQMD 캐시 ({type(error).__name__})"

print("데이터 출처:", oqmd_source)
print("응답 상태:", oqmd_payload["response_message"])

### 4-3. 응답을 표로 바꾸기
응답 안의 `data`는 자료 하나가 딕셔너리 하나인 목록입니다.
이 목록을 그대로 `pd.DataFrame()`에 넣으면 표가 됩니다.
API 응답을 pandas로 넘기는 이 흐름은 앞으로 계속 반복되니 눈에 익혀 두세요.

출력에서 "현재 페이지 행 수"와 "전체 검색 건수"가 다른 점을 확인해 보세요.
`limit`으로 20건만 받았기 때문이며, 표의 행 수를 검색 결과 전체로 착각하면 안 됩니다.

In [ ]:
oqmd_table = pd.DataFrame(oqmd_payload["data"], columns=OQMD_FIELDS)

print("현재 페이지 행 수:", len(oqmd_table))
print("전체 검색 건수:", oqmd_payload["meta"]["data_available"])

## 열이 많아서 눈여겨볼 다섯 개만 골라 표시합니다.
display(oqmd_table[["name", "entry_id", "ntypes", "delta_e", "stability"]])

## 5. 선택: MP에서 LiFePO₄ 한 번 조회하기 · 3분

여기까지가 기본 사전학습입니다.
이 절은 Materials Project(MP) API 키가 있는 학생만 해보면 되고, 키가 없으면 건너뛰어도 수업을 따라오는 데 문제없습니다.

방식은 4절의 OQMD와 거의 같습니다. 다른 점은 MP가 **인증 키를 함께 보내야** 답해 준다는 것뿐입니다.
같은 화학식이라도 구조가 여러 개면 행도 여러 개 나옵니다.

이번에는 최대 10행만 확인하고, 검색 조건을 제대로 다루는 방법과 `MPRester` 사용법은 02 노트북에서 이어가겠습니다.

### 5-1. API 키 입력하기
키는 비밀번호와 같습니다. 코드나 제출 파일에 그대로 적어 두면 공개되어 버리니 조심하세요.
`getpass`는 입력한 글자가 화면에 보이지 않게 받아 줍니다.

키는 [MP 계정 페이지](https://next-gen.materialsproject.org/api)에서 발급받을 수 있습니다.

In [ ]:
from getpass import getpass

## 환경변수에 키를 넣어 둔 경우에는 그것을 쓰고, 없을 때만 직접 입력받습니다.
MP_API_KEY = os.getenv("MP_API_KEY", "").strip()
if not MP_API_KEY:
    MP_API_KEY = getpass("MP API key: ").strip()

### 5-2. 조회하고 표로 보기
OQMD와 달리 키를 `headers`에 담아 함께 보냅니다. 서버는 이 값을 보고 요청한 사람을 확인합니다.

결과 표의 `band_gap`은 eV, `energy_above_hull`은 eV/atom 단위입니다.
`energy_above_hull`이 0에 가까울수록 계산상 안정한 구조입니다.

In [ ]:
## 키를 입력하지 않고 실행했다면 여기서 멈춥니다.
if not MP_API_KEY:
    raise ValueError("MP API 키가 필요합니다. 이 절은 선택 사항이므로 건너뛰어도 됩니다.")

MP_FIELDS = ["material_id", "formula_pretty", "band_gap", "energy_above_hull"]

## X-API-KEY 헤더에 키를 담아 보내는 것이 OQMD와 다른 점입니다.
mp_response = requests.get(
    "https://api.materialsproject.org/materials/summary/",
    headers={"X-API-KEY": MP_API_KEY},
    params={"formula": "LiFePO4", "_fields": ",".join(MP_FIELDS), "_limit": 10},
    timeout=30,
)
mp_response.raise_for_status()

## 4-3과 같은 방식으로 응답 목록을 표로 바꿉니다.
mp_table = pd.DataFrame(mp_response.json()["data"], columns=MP_FIELDS)
print("이번 요청에서 받은 행 수:", len(mp_table))
display(mp_table)